# Phase 3: Feature Extraction

Raw EEG waveforms are hard for classifiers to work with directly. Feature extraction converts each (32, 640) EEG segment into a compact vector of meaningful numbers that capture the signal's characteristics.

We extract 3 categories of features (same as published research on SAM 40):
1) **Frequency-Domain**: Band power in Delta, Theta, Alpha, Beta, Gamma + Beta/Alpha ratio (stress indicator)
       → 6 features × 32 channels = 192 features
2) **Time-Domain**: Hjorth parameters (Activity, Mobility, Complexity) + Statistical features (mean, std, skewness, kurtosis)
       → 7 features × 32 channels = 224 features
3) **Non-Linear**: Spectral entropy, Higuchi fractal dimension
       → 2 features × 32 channels = 64 features

Total: 480 features per segment (before any feature selection)

**Prerequisites**:
- Phase 2 completed (preprocessed_segments.npz + segment_labels.csv)

In [2]:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from scipy.stats import skew, kurtosis
import time
import warnings
warnings.filterwarnings('ignore')
 
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# Configuration

In [3]:

BASE_DIR = os.path.join("..", "")  # If running from Notebooks/
 
DATA_DIR = os.path.join(BASE_DIR, "Results", "phase2")  # Load from Phase 2 output
OUTPUT_DIR = os.path.join(BASE_DIR, "Results", "phase3")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Constants

In [4]:

SAMPLING_RATE = 128
N_CHANNELS = 32
 
CHANNEL_NAMES = [
    'Cz', 'Fz', 'Fp1', 'F7', 'F3', 'FC1', 'C3', 'FC5',
    'FT9', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO9', 'O1',
    'Pz', 'Oz', 'O2', 'PO10', 'P8', 'P4', 'CP2', 'CP6',
    'T8', 'FT10', 'FC6', 'C4', 'FC2', 'F4', 'F8', 'Fp2'
]
 
EEG_BANDS = {
    'Delta':  (0.5,  4),
    'Theta':  (4,    8),
    'Alpha':  (8,   13),
    'Beta':   (13,  30),
    'Gamma':  (30,  45),
}
 
TASK_COLORS = {
    'Relaxation': '#2ecc71', 'Stroop': '#e74c3c',
    'Mirror_Image': '#f39c12', 'Arithmetic': '#3498db',
}

# Feature Extraction Functions

## Frequency Domain Features

Compute power in each EEG frequency band using Welch's PSD method.

The brain produces oscillations at different frequencies, and each frequency range (band) is associated with different mental states:
- **Delta (0.5-4 Hz)**:  Deep sleep. Not very useful for our task since subjects are awake, but good to include.
- **Theta (4-8 Hz)**:    Memory, drowsiness. Can increase during cognitive effort (frontal theta).
- **Alpha (8-13 Hz)**:   Relaxed wakefulness. DECREASES during stress (alpha suppression / desynchronization).
- **Beta (13-30 Hz)**:   Active thinking, focus. INCREASES during stress and cognitive engagement.
- **Gamma (30-45 Hz)**:  High-level processing, concentration.

The Beta/Alpha ratio combines the two key changes (alpha down, beta up) into one powerful stress indicator.

In [5]:


def extract_band_powers(signal, fs=SAMPLING_RATE):
    
    """
    Parameters:
        signal: 1D array of EEG samples for one channel
        fs: sampling rate
    
    Returns:
        dict with 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma', 'Beta_Alpha_Ratio'
    """

    # Compute PSD using Welch's method
    freqs, psd = welch(signal, fs=fs, nperseg=min(256, len(signal)),
                       noverlap=128, scaling='density')
    freq_res = freqs[1] - freqs[0]
    
    powers = {}
    for band_name, (lo, hi) in EEG_BANDS.items():
        idx = np.where((freqs >= lo) & (freqs <= hi))[0]
        powers[band_name] = np.sum(psd[idx]) * freq_res
    
    # Beta/Alpha ratio (avoid division by zero)
    alpha = powers['Alpha']
    beta = powers['Beta']
    powers['Beta_Alpha_Ratio'] = beta / alpha if alpha > 1e-10 else 0.0
    
    return powers

## Time Domain Features

Compute Hjorth parameters: Activity, Mobility, Complexity.

These are simple but powerful features that describe the shape of a time-series signal:
- **Activity**:   Variance of the signal. Measures signal power. Higher activity = larger amplitude fluctuations.
- **Mobility**:   How quickly the signal changes. Computed as the ratio of std of the first derivative to std of the signal. Higher mobility = more frequent oscillations.
- **Complexity**: How much the signal resembles a pure sine wave.A pure sine has complexity = 1. More complex signals (multiple frequencies mixed) have higher complexity. Computed as mobility of the derivative / mobility of signal.

These are computationally cheap (just variance and derivatives) and work well for EEG classification.

In [6]:

def extract_hjorth_parameters(signal):

    """
    Parameters:
        signal: 1D array
    
    Returns:
        dict with 'Activity', 'Mobility', 'Complexity'
    """

    # Activity = variance of the signal
    activity = np.var(signal)
    
    # First derivative (discrete: difference between consecutive samples)
    d1 = np.diff(signal)
    # Second derivative
    d2 = np.diff(d1)
    
    # Mobility = sqrt(var(d1) / var(signal))
    var_d1 = np.var(d1)
    mobility = np.sqrt(var_d1 / activity) if activity > 1e-10 else 0.0
    
    # Complexity = mobility(d1) / mobility(signal)
    var_d2 = np.var(d2)
    mobility_d1 = np.sqrt(var_d2 / var_d1) if var_d1 > 1e-10 else 0.0
    complexity = mobility_d1 / mobility if mobility > 1e-10 else 0.0
    
    return {
        'Activity': activity,
        'Mobility': mobility,
        'Complexity': complexity,
    }


Compute basic statistical features of the signal.

- **Mean**:      Average amplitude (should be ~0 after normalization)
- **Std**:       Spread of amplitudes (signal variability)
- **Skewness**:  Asymmetry of the amplitude distribution (0 = symmetric, positive = right-skewed)
- **Kurtosis**:  "Peakedness" of the distribution (0 = Gaussian, positive = heavy tails/sharp peak)

In [7]:
def extract_statistical_features(signal):

    """
    Parameters:
        signal: 1D array
    Returns:
        dict with 'Mean', 'Std', 'Skewness', 'Kurtosis'
    """
    return {
        'Mean': np.mean(signal),
        'Std': np.std(signal),
        'Skewness': skew(signal),
        'Kurtosis': kurtosis(signal),
    }

## Non Linear Features

Compute spectral entropy to measure the "disorder" in the frequency distribution.

If a signal has all its power concentrated at one frequency (like a pure tone), its spectral entropy is *LOW* (very ordered). If power is spread evenly across all frequencies (like white noise), its spectral entropy is *HIGH* (very disordered).

For EEG:
- Relaxation tends to have Lower spectral entropy (dominated by alpha)
- Stress tends to have Higher spectral entropy (power spread across multiple bands as the brain engages multiple processing networks)

In [8]:


def extract_spectral_entropy(signal, fs=SAMPLING_RATE):
    
    """
    Parameters:
        signal: 1D array
        fs: sampling rate
    
    Returns:
        float: spectral entropy value (0 to 1, normalized)
    """

    freqs, psd = welch(signal, fs=fs, nperseg=min(256, len(signal)), noverlap=128, scaling='density')
    
    # Normalize PSD to get a probability distribution
    psd_norm = psd / (np.sum(psd) + 1e-10)
    
    # Remove zeros to avoid log(0)
    psd_norm = psd_norm[psd_norm > 0]
    
    # Shannon entropy
    entropy = -np.sum(psd_norm * np.log2(psd_norm))
    
    # Normalize by maximum possible entropy (uniform distribution)
    max_entropy = np.log2(len(psd_norm))
    spectral_entropy = entropy / max_entropy if max_entropy > 0 else 0.0
    
    return spectral_entropy


Compute Higuchi's Fractal Dimension to measure signal complexity.

Fractal dimension quantifies how *complex* or *irregular* a signal is. A smooth sine wave has low FD (~1.0), while a very noisy, jagged signal has high FD (approaching 2.0).

For EEG:
- More complex brain activity → higher fractal dimension
- Stress signals tend to have higher FD due to involvement of multiple neural networks simultaneously

Higuchi's method is considered the most accurate FD estimator for time series data. It works by computing the *length* of the signal at different scales and finding how that length scales.

In [9]:

def extract_higuchi_fd(signal, kmax=10):

    """
    Parameters:
        signal: 1D array
        kmax: maximum scale factor (default 10)
    
    Returns:
        float: Higuchi fractal dimension (typically 1.0 to 2.0)
    """
    N = len(signal)
    L = []
    x = np.arange(1, kmax + 1)
    
    for k in range(1, kmax + 1):
        Lk = []
        for m in range(1, k + 1):
            # Construct sub-signal starting at index m, stepping by k
            indices = np.arange(m - 1, N, k)
            if len(indices) < 2:
                continue
            sub = signal[indices]
            
            # Compute length of this sub signal
            Lmk = np.sum(np.abs(np.diff(sub)))
            # Normalize
            n_seg = len(indices) - 1
            Lmk = (Lmk * (N - 1)) / (k * n_seg * k)
            Lk.append(Lmk)
        
        if Lk:
            L.append(np.mean(Lk))
        else:
            L.append(1e-10)
    
    L = np.array(L)
    L[L <= 0] = 1e-10
    
    # Fractal dimension = slope of log(L) vs log(1/k)
    log_k = np.log(1.0 / x)
    log_L = np.log(L)
    
    # Linear regression
    coeffs = np.polyfit(log_k, log_L, 1)
    hfd = coeffs[0]
    
    return hfd

# Main Feature Extraction Pipeline

Extracting all features from every segment

For each of the 2400 segments (shape 32×640):
- For each of the 32 channels, compute:  `6 frequency features + 7 time features + 2 nonlinear features = 15`
- Total:  `15 × 32 = 480 features per segment`

In [10]:

def extract_all_features(segments):

    """
    ArithmeticErrorParameters:
        segments: np.array of shape (n_segments, 32, 640)
    
    Returns:
        feature_matrix: np.array of shape (n_segments, 480)
        feature_names: list of 480 feature name strings
    """
    n_segments = len(segments)
    
    # Build feature names list
    feature_names = []
    freq_feats = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma', 'Beta_Alpha_Ratio']
    time_feats = ['Activity', 'Mobility', 'Complexity', 'Mean', 'Std', 'Skewness', 'Kurtosis']
    nonlinear_feats = ['Spectral_Entropy', 'Higuchi_FD']
    all_feat_names = freq_feats + time_feats + nonlinear_feats
    
    for ch_name in CHANNEL_NAMES:
        for feat_name in all_feat_names:
            feature_names.append(f"{ch_name}_{feat_name}")
    
    n_features = len(feature_names)
    print(f"Features per segment: {n_features}")
    print(f"  Frequency:  {len(freq_feats)} × {N_CHANNELS} channels = {len(freq_feats)*N_CHANNELS}")
    print(f"  Time:       {len(time_feats)} × {N_CHANNELS} channels = {len(time_feats)*N_CHANNELS}")
    print(f"  Nonlinear:  {len(nonlinear_feats)} × {N_CHANNELS} channels = {len(nonlinear_feats)*N_CHANNELS}")
    print(f"  Total:      {n_features}")
    
    # Extract features
    feature_matrix = np.zeros((n_segments, n_features))
    start_time = time.time()
    
    for seg_idx in range(n_segments):
        if (seg_idx + 1) % 200 == 0 or seg_idx == 0:
            elapsed = time.time() - start_time
            rate = (seg_idx + 1) / elapsed if elapsed > 0 else 0
            remaining = (n_segments - seg_idx - 1) / rate if rate > 0 else 0
            print(f"  [{seg_idx+1}/{n_segments}] "
                  f"({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)")
        
        segment = segments[seg_idx]  # (32, 640)
        feat_idx = 0
        
        for ch in range(N_CHANNELS):
            signal = segment[ch]  # (640,)
            
            # Frequency features:
            band_powers = extract_band_powers(signal)
            for feat_name in freq_feats:
                feature_matrix[seg_idx, feat_idx] = band_powers[feat_name]
                feat_idx += 1
            
            # Time domain features:
            hjorth = extract_hjorth_parameters(signal)
            stats = extract_statistical_features(signal)
            for feat_name in time_feats:
                if feat_name in hjorth:
                    feature_matrix[seg_idx, feat_idx] = hjorth[feat_name]
                else:
                    feature_matrix[seg_idx, feat_idx] = stats[feat_name]
                feat_idx += 1
            
            # Nonlinear features:
            feature_matrix[seg_idx, feat_idx] = extract_spectral_entropy(signal)
            feat_idx += 1
            feature_matrix[seg_idx, feat_idx] = extract_higuchi_fd(signal)
            feat_idx += 1
    
    total_time = time.time() - start_time
    print(f"\nFeature extraction complete in {total_time:.1f}s")
    print(f"Feature matrix shape: {feature_matrix.shape}")
    
    # Check for NaN or Inf values
    n_nan = np.isnan(feature_matrix).sum()
    n_inf = np.isinf(feature_matrix).sum()
    if n_nan > 0 or n_inf > 0:
        print(f"  WARNING: {n_nan} NaN values, {n_inf} Inf values found")
        # Replace with 0
        feature_matrix = np.nan_to_num(feature_matrix, nan=0.0, posinf=0.0, neginf=0.0)
        print(f"  Replaced with 0.0")
    else:
        print(f"  ✓ No NaN or Inf values")
    
    return feature_matrix, feature_names

# Visualization

In [12]:


#How key features differ between stress levels:

def plot_1_feature_distributions(feature_matrix, feature_names, seg_labels_df, output_dir):
    print("\nPlot 1: Feature distributions by stress level")
    
    # Pick a few important features to visualize
    key_features = [
        'Fz_Alpha',            # Alpha power at frontal midline
        'Fz_Beta',             # Beta power at frontal midline
        'Fz_Beta_Alpha_Ratio', # Stress indicator
        'Cz_Activity',         # Signal power at central midline
        'Pz_Spectral_Entropy', # Spectral complexity at parietal
        'Fz_Higuchi_FD',       # Fractal dimension at frontal
    ]
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Feature Distributions by Stress Level\n'
                 'How well can each feature separate the classes?',
                 fontsize=13, fontweight='bold')
    
    stress_colors = {'Relaxed': '#2ecc71', 'Low Stress': '#f39c12', 'High Stress': '#e74c3c'}
    
    for idx, feat_name in enumerate(key_features):
        ax = axes[idx // 3, idx % 3]
        
        if feat_name in feature_names:
            feat_idx = feature_names.index(feat_name)
            
            for level in ['Relaxed', 'Low Stress', 'High Stress']:
                mask = seg_labels_df['stress_level'] == level
                values = feature_matrix[mask, feat_idx]
                ax.hist(values, bins=30, alpha=0.5, label=level,
                        color=stress_colors[level], edgecolor='white', density=True)
            
            ax.set_xlabel(feat_name.replace('_', ' '))
            ax.set_ylabel('Density')
            ax.legend(fontsize=8)
            ax.set_title(feat_name, fontweight='bold', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '01_feature_distributions.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 01_feature_distributions.png")

In [13]:

# Correlation between feature categories to identify redundancy.

def plot_2_feature_correlation(feature_matrix, feature_names, output_dir):
    print("\nPlot 2: Feature correlation heatmap")
    
    # Picking one channel (Fz here) and showing correlations between all feature types
    fz_features = [f for f in feature_names if f.startswith('Fz_')]
    fz_indices = [feature_names.index(f) for f in fz_features]
    fz_labels = [f.replace('Fz_', '') for f in fz_features]
    
    fz_data = feature_matrix[:, fz_indices]
    corr = np.corrcoef(fz_data.T)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(fz_labels)))
    ax.set_yticks(range(len(fz_labels)))
    ax.set_xticklabels(fz_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(fz_labels, fontsize=8)
    ax.set_title('Feature Correlation Matrix — Channel Fz\n'
                 'Red = correlated, Blue = anti-correlated',
                 fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Correlation')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '02_feature_correlation.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 02_feature_correlation.png")

In [14]:

# Rank features by how well they separate stress levels using ANOVA F-statistic (higher = better separation)

def plot_3_top_features(feature_matrix, feature_names, seg_labels_df, output_dir):
    print("\nPlot 3: Top discriminative features")
    
    from scipy.stats import f_oneway
    
    # Getting indices for each stress level
    labels = seg_labels_df['stress_level'].values
    groups = {}
    for level in ['Relaxed', 'Low Stress', 'High Stress']:
        groups[level] = feature_matrix[labels == level]
    
    # Compute F-statistic for each feature
    f_scores = []
    for i in range(len(feature_names)):
        try:
            f_val, p_val = f_oneway(groups['Relaxed'][:, i],
                                     groups['Low Stress'][:, i],
                                     groups['High Stress'][:, i])
            f_scores.append(f_val if not np.isnan(f_val) else 0)
        except:
            f_scores.append(0)
    
    f_scores = np.array(f_scores)
    
    # Get top 20 features
    top_idx = np.argsort(f_scores)[::-1][:20]
    top_names = [feature_names[i] for i in top_idx]
    top_scores = f_scores[top_idx]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Color bars by feature type
    colors = []
    for name in top_names:
        if any(x in name for x in ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma', 'Ratio']):
            colors.append('#3498db')  # Frequency features
        elif any(x in name for x in ['Activity', 'Mobility', 'Complexity', 'Mean', 'Std', 'Skew', 'Kurt']):
            colors.append('#2ecc71')  # Time features
        else:
            colors.append('#e74c3c')  # Nonlinear features
    
    bars = ax.barh(range(len(top_names)), top_scores, color=colors, alpha=0.8, edgecolor='white')
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels([n.replace('_', ' ') for n in top_names], fontsize=9)
    ax.set_xlabel('ANOVA F-Score (higher = better class separation)')
    ax.set_title('Top 20 Most Discriminative Features for Stress Classification\n'
                 'Blue = frequency, Green = time, Red = nonlinear',
                 fontweight='bold')
    ax.invert_yaxis()
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#3498db', label='Frequency domain'),
        Patch(facecolor='#2ecc71', label='Time domain'),
        Patch(facecolor='#e74c3c', label='Nonlinear'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '03_top_features.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ Saved: 03_top_features.png")
    
    # Print top 10
    print("\n  Top 10 most discriminative features:")
    for i in range(min(10, len(top_names))):
        print(f"    {i+1:2d}. {top_names[i]:30s}  F-score: {top_scores[i]:.1f}")

# MAIN

In [15]:

if __name__ == "__main__":
    print("=" * 60)
    print("  EEG COGNITIVE STRESS CLASSIFICATION")
    print("  Phase 3: Feature Extraction")
    print("=" * 60)

  EEG COGNITIVE STRESS CLASSIFICATION
  Phase 3: Feature Extraction


In [16]:

# Loading Preprocessed Data

print(f"\n▶ Loading preprocessed data from Phase 2...")
    
segments_path = os.path.join(DATA_DIR, 'preprocessed_segments.npz')
labels_path = os.path.join(DATA_DIR, 'segment_labels.csv')
    
if not os.path.exists(segments_path):
    print(f"  ✗ Not found: {segments_path}")
    print(f"  Run Phase 2 first.")
    exit(1)
    
data = np.load(segments_path)
segments = data['segments']
seg_labels_df = pd.read_csv(labels_path)
    
print(f"  Loaded {len(segments)} segments, shape: {segments[0].shape}")
print(f"  Labels: {len(seg_labels_df)} rows")


▶ Loading preprocessed data from Phase 2...
  Loaded 2400 segments, shape: (32, 640)
  Labels: 2400 rows


In [18]:

# Extracting features

print(f"\n▶ Extracting features...")
    
feature_matrix, feature_names = extract_all_features(segments)


▶ Extracting features...
Features per segment: 480
  Frequency:  6 × 32 channels = 192
  Time:       7 × 32 channels = 224
  Nonlinear:  2 × 32 channels = 64
  Total:      480
  [1/2400] (0s elapsed, ~0s remaining)
  [200/2400] (14s elapsed, ~149s remaining)
  [400/2400] (26s elapsed, ~132s remaining)
  [600/2400] (39s elapsed, ~117s remaining)
  [800/2400] (52s elapsed, ~104s remaining)
  [1000/2400] (65s elapsed, ~91s remaining)
  [1200/2400] (78s elapsed, ~78s remaining)
  [1400/2400] (98s elapsed, ~70s remaining)
  [1600/2400] (120s elapsed, ~60s remaining)
  [1800/2400] (141s elapsed, ~47s remaining)
  [2000/2400] (169s elapsed, ~34s remaining)
  [2200/2400] (183s elapsed, ~17s remaining)
  [2400/2400] (200s elapsed, ~0s remaining)

Feature extraction complete in 199.7s
Feature matrix shape: (2400, 480)
  ✓ No NaN or Inf values


In [19]:
# Visualizing Features

print(f"\n▶ Generating visualizations...")
plot_1_feature_distributions(feature_matrix, feature_names, seg_labels_df, OUTPUT_DIR)
plot_2_feature_correlation(feature_matrix, feature_names, OUTPUT_DIR)
plot_3_top_features(feature_matrix, feature_names, seg_labels_df, OUTPUT_DIR)


▶ Generating visualizations...

Plot 1: Feature distributions by stress level
  ✓ Saved: 01_feature_distributions.png

Plot 2: Feature correlation heatmap
  ✓ Saved: 02_feature_correlation.png

Plot 3: Top discriminative features
  ✓ Saved: 03_top_features.png

  Top 10 most discriminative features:
     1. Fp1_Alpha                       F-score: 54.4
     2. P4_Alpha                        F-score: 52.3
     3. CP2_Alpha                       F-score: 47.2
     4. P3_Alpha                        F-score: 46.0
     5. Fz_Alpha                        F-score: 44.6
     6. P8_Alpha                        F-score: 43.7
     7. Oz_Alpha                        F-score: 42.6
     8. CP1_Alpha                       F-score: 40.5
     9. CP2_Activity                    F-score: 40.5
    10. CP2_Std                         F-score: 39.8


In [20]:
# Save Features

print(f"\n▶ Saving feature matrix...")
    
# Save as .npz for fast loading in Phase 4
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'features.npz'),
    feature_matrix=feature_matrix,
    feature_names=np.array(feature_names),
)
    
# Also save as CSV for easy inspection (warning: large file)
# Only save a summary version with feature names as columns
feature_df = pd.DataFrame(feature_matrix, columns=feature_names)
feature_df = pd.concat([seg_labels_df, feature_df], axis=1)
feature_df.to_csv(os.path.join(OUTPUT_DIR, 'features_with_labels.csv'), index=False)
    
npz_size = os.path.getsize(os.path.join(OUTPUT_DIR, 'features.npz')) / (1024*1024)
csv_size = os.path.getsize(os.path.join(OUTPUT_DIR, 'features_with_labels.csv')) / (1024*1024)
    
print(f"  ✓ features.npz ({npz_size:.1f} MB)")
print(f"  ✓ features_with_labels.csv ({csv_size:.1f} MB)")


▶ Saving feature matrix...
  ✓ features.npz (8.4 MB)
  ✓ features_with_labels.csv (21.4 MB)


# Summary

In [21]:
print(f"\n{'='*60}")
print(f"  PHASE 3 COMPLETE!")
print(f"{'='*60}")
print(f"  Segments:         {len(segments)}")
print(f"  Features/segment: {len(feature_names)}")
print(f"  Feature matrix:   {feature_matrix.shape}")
print(f"  Feature types:")
print(f"    Frequency:   6 × 32 ch = 192 (band powers + beta/alpha ratio)")
print(f"    Time:        7 × 32 ch = 224 (Hjorth + statistical)")
print(f"    Nonlinear:   2 × 32 ch =  64 (spectral entropy + Higuchi FD)")
print(f"    Total:       {len(feature_names)}")
print(f"\n  Outputs in: {os.path.abspath(OUTPUT_DIR)}/")
print(f"    01_feature_distributions.png")
print(f"    02_feature_correlation.png")
print(f"    03_top_features.png")
print(f"    features.npz")
print(f"    features_with_labels.csv")
print(f"\n  Next → Phase 4: Classification")
print(f"{'='*60}")


  PHASE 3 COMPLETE!
  Segments:         2400
  Features/segment: 480
  Feature matrix:   (2400, 480)
  Feature types:
    Frequency:   6 × 32 ch = 192 (band powers + beta/alpha ratio)
    Time:        7 × 32 ch = 224 (Hjorth + statistical)
    Nonlinear:   2 × 32 ch =  64 (spectral entropy + Higuchi FD)
    Total:       480

  Outputs in: c:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase3/
    01_feature_distributions.png
    02_feature_correlation.png
    03_top_features.png
    features.npz
    features_with_labels.csv

  Next → Phase 4: Classification
